# Week 1 · Notebook 6 — Mean Reversion Strategy

This notebook implements the strategy shown in the TikTok video, applied to every
stock in our trading universe **simultaneously**.

### The rule (per stock, every trading day)

| Last week's return | Action |
|---|---|
| ≤ −5 % | **BUY** \$5 of that stock (average down) |
| ≥ +10 % | **SELL** \$10 of that stock (take profit) |
| anything else | hold unchanged |

Apply this to **all** stocks at once — in the video ~4,500 stocks, here every
stock in `data/egx/`.

### Why this is mean reversion

Mean reversion is the idea that extreme moves tend to unwind. A stock that fell
hard last week was probably oversold — buying it bets on the bounce back toward
its average. A stock that surged 10 % may be overbought — selling reduces exposure
before it cools off. The same logic drives RSI-based strategies; here the signal
is simply the raw 5-day return.

### The translation challenge

Our simulator speaks **weights** (fractions of the portfolio that sum to 1), not
raw dollars. The `SampleStrategy` class keeps a dollar ledger per stock and
normalises it to weights each day — so the accounting is honest, and the strategy
plugs straight into the same `run_backtest` engine every other strategy uses.

## 0. Setup

In [ ]:
import sys, os
# Walk up until we find the project root (the folder that contains 'src/').
while not os.path.isdir('src') and os.path.dirname(os.getcwd()) != os.getcwd():
    os.chdir('..')
sys.path.insert(0, 'src')

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

from tradinglab.data_feed import DataFeed
from tradinglab.simulator import PortfolioSimulator
from tradinglab.backtester import run_backtest
from tradinglab.strategies.sample_strategy import SampleStrategy, DISPLAY_NAME
import tradinglab.metrics as metrics

print('Strategy display name:', DISPLAY_NAME)

## 1. Load the universe

We use the full EGX universe — every CSV in `data/egx/`. This mirrors the
TikTok example where the rule runs across thousands of stocks at once.

In [ ]:
feed = DataFeed.from_dir('data/egx')
print(f'{feed.n_assets} stocks  ·  {feed.n_days} trading days')
print(f'Date range: {feed.dates[0].date()} → {feed.dates[-1].date()}')
print('Symbols:', feed.symbols)

## 2. Understand the signal: 5-day return

"Last week" = the last 5 trading days. The feature `return_5d` (feature index 6
in our observation tensor) already computes this for every stock.

Let's look at the distribution on a recent day to see how many stocks would
trigger each side of the rule.

In [ ]:
from tradinglab.features import feature_columns

# Grab today's 5-day return for every stock (last row of the feature matrix).
ret5 = np.array([
    feature_columns(feed, a)[-1, 6]   # index 6 = return_5d
    for a in range(feed.n_assets)
])
ret5 = ret5[~np.isnan(ret5)]   # drop any NaN (stocks with insufficient history)

buy_count  = (ret5 <= -0.05).sum()
sell_count = (ret5 >=  0.10).sum()
hold_count = len(ret5) - buy_count - sell_count

print(f'On the most recent day:')
print(f'  BUY  signal (≤ -5 %): {buy_count} stocks')
print(f'  SELL signal (≥ +10%): {sell_count} stocks')
print(f'  HOLD (no signal)    : {hold_count} stocks')

fig, ax = plt.subplots(figsize=(9, 3))
ax.hist(ret5 * 100, bins=30, color='#5b9dfa', edgecolor='#1b2438')
ax.axvline(-5,  color='#4ade80', linewidth=1.5, linestyle='--', label='Buy threshold  −5%')
ax.axvline(10, color='#f87171', linewidth=1.5, linestyle='--', label='Sell threshold +10%')
ax.set_xlabel('5-day return (%)')
ax.set_ylabel('Number of stocks')
ax.set_title('Distribution of last-week returns across the universe')
ax.legend()
ax.grid(alpha=0.25)
plt.tight_layout()
plt.show()

## 3. How the strategy works (step by step)

The `SampleStrategy` class is stateful — it remembers how many dollars it holds in
each stock across days. Here we trace through 3 days manually on a tiny 3-stock
universe to see exactly what happens.

In [ ]:
# Fake observations for 3 stocks.  Shape: (n_assets, lookback, n_features)
# We only use feature index 6 (return_5d) — everything else is zero.
N_FEATURES = 9

def make_obs(ret5d_values):
    """Build a minimal observation where return_5d = ret5d_values (one per stock)."""
    n = len(ret5d_values)
    obs = np.zeros((n, 1, N_FEATURES))
    obs[:, 0, 6] = ret5d_values   # index 6 = return_5d
    return obs

strategy = SampleStrategy(n_assets=3)

scenarios = [
    # (day_label, [ret5d_stock0, ret5d_stock1, ret5d_stock2])
    ('Day 1', [-0.06,  0.00,  0.12]),   # stock 0 down 6% → BUY; stock 2 up 12% → SELL (no position yet, nothing to sell)
    ('Day 2', [-0.07,  0.02, -0.05]),   # stock 0 down again → add BUY; stock 2 down 5% → BUY
    ('Day 3', [ 0.03,  0.11,  0.15]),   # stock 1 up 11% → SELL (no position); stock 2 up 15% → SELL
]

print(f'{"Day":<8} {"ret5d":>22}   {"Dollar ledger after":>30}   {"Weights":>30}')
print('-' * 100)
for label, ret5d in scenarios:
    obs     = make_obs(ret5d)
    weights = strategy(obs)
    dollars = strategy._dollars.copy()
    ret_str = '  '.join(f'{r*100:+.0f}%' for r in ret5d)
    dol_str = '  '.join(f'${d:.1f}' for d in dollars)
    wgt_str = '  '.join(f'{w:.2f}' for w in weights)
    print(f'{label:<8} [{ret_str}]   [{dol_str}]   [{wgt_str}]')

## 4. Run the full backtest

Now run against the real EGX data. The engine calls the strategy once per trading
day, for every day in history — exactly like the simulator would run 4,500 stocks
in the TikTok example.

In [ ]:
sim = PortfolioSimulator(feed, benchmark='equal_weight')

# Always create a fresh SampleStrategy so the dollar ledger starts at zero.
# lookback=10 gives the observation window enough depth to compute return_5d.
strategy = SampleStrategy(feed.n_assets)
result   = run_backtest(sim, strategy, lookback=10)

print('Backtest complete.')
print(f'  Days traded : {len(result["portfolio_returns"])}')
print(f'  Date range  : {result["dates"][0].date()} → {result["dates"][-1].date()}')

## 5. Equity curve

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))

ax.plot(result['dates'], result['portfolio'] * 1000,
        color='#5b9dfa', linewidth=2, label=DISPLAY_NAME)
ax.plot(result['dates'], result['benchmark'] * 1000,
        color='#f0a857', linewidth=1.5, linestyle='--', label='Equal-Weight Benchmark')

ax.set_title(f'{DISPLAY_NAME} — equity curve (growth of 1,000 EGP)', fontsize=13)
ax.set_xlabel('Date')
ax.set_ylabel('Portfolio value (EGP)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'{v:,.0f}'))
ax.legend()
ax.grid(alpha=0.25)
plt.tight_layout()
plt.show()

## 6. Performance metrics

In [ ]:
rets  = result['portfolio_returns']
brets = result['benchmark_returns']

rows = [
    ('Total return',      f"{metrics.total_return(rets)*100:+.1f}%",  f"{metrics.total_return(brets)*100:+.1f}%"),
    ('Annualised return', f"{metrics.annualized_return(rets)*100:+.1f}%", f"{metrics.annualized_return(brets)*100:+.1f}%"),
    ('Volatility (ann.)', f"{metrics.volatility(rets)*100:.1f}%",     f"{metrics.volatility(brets)*100:.1f}%"),
    ('Sharpe ratio',      f"{metrics.sharpe(rets):.2f}",              f"{metrics.sharpe(brets):.2f}"),
    ('Max drawdown',      f"{metrics.max_drawdown(rets)*100:.1f}%",   f"{metrics.max_drawdown(brets)*100:.1f}%"),
]

col_w = 28
strat_col  = 'TikTok Strategy'
bench_col  = 'Benchmark (equal-wt)'
header = f"{'Metric':<25} {strat_col:>{col_w}} {bench_col:>{col_w}}"
print(header)
print('-' * len(header))
for metric, strat_val, bench_val in rows:
    print(f"{metric:<25} {strat_val:>{col_w}} {bench_val:>{col_w}}")

## 7. Signal activity — how many stocks trade each day?

In [ ]:
from tradinglab.observation import build_observation

lookback   = 10
start      = lookback
end        = feed.n_days - 1

daily_buys  = []
daily_sells = []

for t in range(start, end):
    obs    = build_observation(feed, t, lookback)
    ret5d  = obs[:, -1, 6]
    daily_buys.append( (ret5d <= -0.05).sum() )
    daily_sells.append((ret5d >=  0.10).sum() )

dates_plot = feed.dates[start:end]

fig, ax = plt.subplots(figsize=(12, 3))
ax.fill_between(dates_plot, daily_buys,  alpha=0.6, color='#4ade80', label='Buy signals')
ax.fill_between(dates_plot, [-s for s in daily_sells], alpha=0.6, color='#f87171', label='Sell signals (mirrored)')
ax.axhline(0, color='#8a94a6', linewidth=0.8)
ax.set_title('Number of buy / sell signals fired each day across the universe')
ax.set_ylabel('Stocks triggered')
ax.legend()
ax.grid(alpha=0.2)
plt.tight_layout()
plt.show()

print(f'Average daily buy  signals: {np.mean(daily_buys):.1f} stocks')
print(f'Average daily sell signals: {np.mean(daily_sells):.1f} stocks')

## 8. Experiment — tune the thresholds

The original rule uses −5 % / +10 %. Try changing the thresholds and see how the
equity curve responds. This is the same kind of sensitivity analysis you'd run
before deploying any strategy.

In [ ]:
configs = [
    ('Original  (−5% / +10%)', -0.05,  0.10),
    ('Tighter   (−3% / +7%)',  -0.03,  0.07),
    ('Looser    (−8% / +15%)', -0.08,  0.15),
]

fig, ax = plt.subplots(figsize=(12, 5))
colors  = ['#5b9dfa', '#a78bfa', '#34d399']

for (label, buy_thr, sell_thr), color in zip(configs, colors):
    strat  = SampleStrategy(feed.n_assets, buy_threshold=buy_thr, sell_threshold=sell_thr)
    res    = run_backtest(sim, strat, lookback=10)
    sr     = metrics.sharpe(res['portfolio_returns'])
    ax.plot(res['dates'], res['portfolio'] * 1000,
            color=color, linewidth=1.8, linestyle='-', label=f'{label}  (Sharpe {sr:.2f})')

ax.plot(result['dates'], result['benchmark'] * 1000,
        color='#f0a857', linewidth=1.5, linestyle='--', label='Equal-Weight Benchmark')

ax.set_title(f'{DISPLAY_NAME} — threshold sensitivity', fontsize=13)
ax.set_ylabel('Portfolio value (EGP)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'{v:,.0f}'))
ax.legend(fontsize=9)
ax.grid(alpha=0.25)
plt.tight_layout()
plt.show()

## 9. What you built

- **`SampleStrategy`** — a stateful class that keeps a dollar ledger across days
  and converts it to weights the simulator can consume.
- **`/backtest/sample_strategy`** — a dashboard endpoint that runs this strategy
  and returns equity curves ready to plot.
- **Display name** — everywhere the strategy appears in a chart or UI it reads
  **"Mean Reversion Strategy"**; the internal code name is `sample_strategy`.

The key insight is that the *exact same rule* used on 4,500 stocks simultaneously
in the TikTok video runs here across the full EGX universe with zero loop changes —
because the observation tensor already has one row per stock and NumPy operations
apply to all of them in a single line.